# Độ tập trung doanh thu: "20% khách mang 80% doanh thu?" và ngưỡng khách VIP

**Bối cảnh.** Head of CRM của cửa hàng quà tặng online (Online Retail II) chuẩn bị chương trình khách VIP cho năm
2012: quản lý tài khoản riêng, giá sỉ ưu đãi. Ngân sách đủ cho khoảng 400–500 khách. Câu hỏi: *"Có đúng là 20% khách
mang 80% doanh thu không? Đặt ngưỡng VIP ở đâu, nhóm đó mang bao nhiêu doanh thu, và có rủi ro gì?"*

Dữ liệu: bộ `online-retail-ii`, sheet `Year 2010-2011` (541 910 dòng, 01/12/2010–09/12/2011). Định nghĩa dùng
chung của track: dòng hợp lệ là bỏ hoá đơn bắt đầu bằng `C`, bỏ dòng `Quantity <= 0` hoặc `Price <= 0`; doanh thu
= `Quantity * Price` (doanh thu gộp); khách là `Customer ID` khác null.

Việc của bạn:

1. Viết bốn hàm đo độ tập trung (được chấm).
2. Phân tích tự do trong các cell `explore`: Pareto, Lorenz, Gini, các ngưỡng VIP ứng viên, và các bẫy (khách mới
   có ít thời gian mua, một khách chiếm phần lớn, khách vãng lai).
3. Viết `starter/MEMO.md` cho Head of CRM theo khung có sẵn (200-600 từ).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) bị bỏ qua khi chấm.
- Chấm phần tự động: `uv run pytest tracks/data-analyst/modules/07-descriptive-eda/exercises/advanced/02-revenue-concentration -q`. Nội dung memo chấm theo rubric trong `SOLUTION.md`.

In [ ]:
# explore: đọc sheet 2010-2011; lần đầu mất 20-90 giây, sau đó đọc bản parquet trong thư mục tạm
import os
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd


def read_retail_1011() -> pd.DataFrame:
    xlsx = ensure("online-retail-ii") / "online_retail_II.xlsx"
    stat = xlsx.stat()
    cache = Path(tempfile.gettempdir()) / "da-cache" / f"retail-1011-{stat.st_size}-{int(stat.st_mtime)}.parquet"
    if cache.exists():
        return pd.read_parquet(cache)
    df = pd.read_excel(xlsx, sheet_name="Year 2010-2011")
    df["Invoice"] = df["Invoice"].astype(str)
    df["StockCode"] = df["StockCode"].astype(str)
    df["Description"] = df["Description"].astype("string")
    cache.parent.mkdir(exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
    df.to_parquet(tmp)
    tmp.replace(cache)
    return df


tx = read_retail_1011()
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Bốn hàm đo độ tập trung

- `customer_revenue(tx) -> pd.Series`: doanh thu mỗi khách trên dòng hợp lệ có `Customer ID`; index tên
  `Customer ID`, Series tên `revenue`, **sắp giảm dần**. `Invoice` có thể là `int` hoặc `str` (đổi sang `str` trước
  khi xét tiền tố `C`).
- `lorenz_curve(x) -> DataFrame`: hai cột `customer_share`, `revenue_share`, có `n + 1` dòng bắt đầu từ `(0, 0)`
  và kết thúc ở `(1, 1)`; sắp giá trị **tăng dần**, dòng thứ `i` là tỉ lệ `i / n` đơn vị nhỏ nhất và tỉ trọng doanh
  thu cộng dồn của chúng.
- `gini(x) -> float`: theo công thức ở phần 3 lý thuyết (0 khi mọi giá trị bằng nhau; với một giá trị dương và
  `n - 1` giá trị 0 thì bằng `(n - 1) / n`).
- `top_share(x, q) -> float`: tỉ trọng doanh thu của `ceil(q * n)` giá trị lớn nhất; `ValueError` nếu `q` ngoài
  `(0, 1]`.

`lorenz_curve`, `gini`, `top_share` nhận list, mảng hoặc Series; `ValueError` nếu rỗng, có NaN, có số âm, hoặc tổng
bằng 0.

In [ ]:
def customer_revenue(tx: pd.DataFrame) -> pd.Series:
    """Doanh thu gộp mỗi khách trên dòng hợp lệ, sắp giảm dần (index "Customer ID", tên "revenue")."""
    # TODO
    raise NotImplementedError


def lorenz_curve(x) -> pd.DataFrame:
    """customer_share, revenue_share: n + 1 điểm từ (0, 0) tới (1, 1), giá trị sắp tăng dần."""
    # TODO
    raise NotImplementedError


def gini(x) -> float:
    """Hệ số Gini của các giá trị không âm."""
    # TODO
    raise NotImplementedError


def top_share(x, q: float) -> float:
    """Tỉ trọng tổng của ceil(q * n) giá trị lớn nhất."""
    # TODO
    raise NotImplementedError

## 2. Phân tích

Khung gợi ý (thêm cell tuỳ ý):

1. **Pareto**: top 1%, 10%, 20% khách mang bao nhiêu doanh thu? Cần bao nhiêu % khách để đạt 80%?
2. **Lorenz và Gini**: vẽ đường Lorenz với đường chéo.
3. **Ngưỡng VIP**: so vài ngưỡng (p90, p95, một mức doanh thu tròn); với mỗi ngưỡng: số khách, doanh thu nhóm, tỉ
   trọng. Ngân sách 400–500 khách.
4. **Bẫy**: khách mua lần đầu vào nửa cuối năm có ít thời gian để chi (so Gini trên cohort mua lần đầu trước
   2011); vài khách lớn nhất là ai, ở nước nào, mua đều hay một lần; khách vãng lai (`Customer ID` trống) chiếm
   bao nhiêu doanh thu và có nằm trong phân tích không.

In [ ]:
# explore: Pareto, Lorenz, Gini
rev = customer_revenue(tx)
print(f"{len(rev):,} khách, doanh thu {rev.sum():,.0f} GBP; gini {gini(rev):.3f}")
for q in (0.01, 0.1, 0.2, 0.5):
    print(f"top {q:.0%} khách: {top_share(rev, q):.1%} doanh thu")
curve = lorenz_curve(rev)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(curve.customer_share, curve.revenue_share, label="Lorenz")
ax.plot([0, 1], [0, 1], color="#9aa5b1", ls="--", label="mọi khách bằng nhau")
ax.set_xlabel("tỉ lệ khách (nhỏ → lớn)")
ax.set_ylabel("tỉ lệ doanh thu cộng dồn")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# explore: chỗ của bạn (ngưỡng VIP, cohort, khách lớn nhất, khách vãng lai)

## 3. Viết memo

Mở `starter/MEMO.md`, điền theo khung. Head of CRM đọc trong hai phút và quyết định ngưỡng; mọi con số lấy từ
notebook.